# Bước 3. Data Analysis with SQL: trả lời câu hỏi mô tả bằng truy vấn

Bước này nạp tệp gốc, bảng mô hình của Bước 2 và bảng mã nước vào một cơ sở dữ liệu DuckDB, rồi dùng SQL để trả lời 19 câu hỏi. Môn ADY201m coi truy vấn dữ liệu là kỹ năng chính, nên các câu hỏi đi từ kiểm tra dữ liệu, tới hai lớp và các lối tắt, phân bố, địa lý, cách chia dữ liệu, rồi tới ba câu hỏi nghiên cứu.
SQL hợp với các câu hỏi dạng "đếm, cộng, nhóm, xếp hạng, lọc theo điều kiện". Phần mô hình hoá để dành cho Bước 6.
Sau mỗi truy vấn có một phép tính lại bằng pandas để đối chiếu: hai cách độc lập phải ra cùng con số.

**Vì sao DuckDB mà không phải SQLite.** DuckDB là cơ sở dữ liệu nhúng trong Python giống SQLite, nhưng đọc thẳng tệp parquet của Bước 2 và có sẵn các hàm thống kê (`MEDIAN`, `QUANTILE_CONT`) cùng các cú pháp `FILTER`, `QUALIFY`, `UNPIVOT`, `GROUP BY ALL`. Cả 19 truy vấn chạy trong vài giây trên 235.370 dòng.

Bảng dưới liệt kê 19 câu hỏi, góc nhìn và kỹ thuật SQL chính của từng câu.

| STT | Câu hỏi | Góc nhìn | Kỹ thuật SQL chính | Tệp kết quả |
|---|---|---|---|---|
| 1 | Dữ liệu có đủ và đúng kích thước không? | Kiểm tra | `COUNT(DISTINCT)`, `GROUP BY ROLLUP`, `GROUPING` | `sql_01_overview.csv` |
| 2 | Địa chỉ trùng thuộc lớp nào, khác nhau ở đâu? | Kiểm tra | bảng tạm `WITH`, `HAVING`, `CASE` | `sql_02_duplicates.csv` |
| 3 | Hai lớp khác nhau thế nào? | Hai lớp | `AVG` của điều kiện, `MEDIAN` | `sql_03_two_classes.csv` |
| 4 | Địa chỉ bắt đầu bằng tiền tố nào? | Lối tắt | `LIKE`, `CASE`, `COUNT(*) FILTER` | `sql_04_prefix_shapes.csv` |
| 5 | Một quy tắc viết tay đạt bao nhiêu? | Lối tắt | `UNION ALL`, `AVG(...) FILTER` | `sql_05_one_line_rules.csv` |
| 6 | Chỉ số tương đồng tách hai lớp tới đâu? | Lối tắt | `FLOOR`, `SUM() OVER` tích luỹ | `sql_06_similarity_bands.csv` |
| 7 | TLD nào đông nhất? | Phân bố | `RANK() OVER`, `QUALIFY`, tỷ trọng tích luỹ | `sql_07_tld_rank.csv` |
| 8 | Ba loại TLD khác nhau thế nào? | Địa lý | `LEFT JOIN` với bảng mã nước | `sql_08_tld_type.csv` |
| 9 | Nước nào (theo TLD) nhiều lừa đảo nhất? | Địa lý | `JOIN`, `HAVING`, `DENSE_RANK`, `NTILE` | `sql_09_countries.csv` |
| 10 | Lừa đảo tập trung ở tên miền nào? | Phân bố | `ROW_NUMBER`, `QUALIFY`, truy vấn con | `sql_10_registered_domains.csv` |
| 11 | Độ dài địa chỉ phân bố thế nào? | Phân bố | `CASE` chia khoảng, `SUM() OVER` theo cột | `sql_11_length_bands.csv` |
| 12 | Phân vị độ dài ở nhóm trang chủ? | Phân bố | `QUANTILE_CONT`, `GROUP BY ALL` | `sql_12_length_quantiles.csv` |
| 13 | Trang lừa đảo có thật sự trống không? | Hai lớp | nhiều điều kiện `AND` trong `AVG` | `sql_13_empty_pages.csv` |
| 14 | Cột `URLLength` lệch theo nguồn nào? | Kiểm tra | hàm chuỗi `LENGTH`, `FILTER` | `sql_14_length_gap.csv` |
| 15 | Chia ngẫu nhiên để lộ tên miền bao nhiêu? | Cách chia | `UNPIVOT`, `LEFT JOIN` nửa | `sql_15_split_domain_overlap.csv` |
| 16 | Hai phần của phép thử TLD chưa thấy khác nhau ra sao? | Cách chia | `AVG(...) FILTER` theo lớp | `sql_16_tld_seen_unseen.csv` |
| 17 | Câu hỏi nghiên cứu 1: hai lớp khác nhau thế nào ở nhóm trang chủ? | Câu hỏi 1 | `WHERE` trên nhóm khó | `sql_17_rq1_homepage_classes.csv` |
| 18 | Câu hỏi nghiên cứu 2: mốc không cần huấn luyện đạt bao nhiêu ở năm seed? | Câu hỏi 2 | `UNPIVOT`, `STDDEV_SAMP`, `UNION ALL` | `sql_18_rq2_rule_by_seed.csv` |
| 19 | Câu hỏi nghiên cứu 3: lối tắt còn đứng vững trong các nhóm khó không? | Câu hỏi 3 | `UNION ALL` các nhóm con, `FILTER` | `sql_19_rq3_rules_by_group.csv` |

**Phạm vi:** mọi truy vấn chạy trên bảng `urls` (235.370 dòng đã bỏ trùng của Bước 2), trừ truy vấn 2 chạy trên bảng `raw_urls` (tệp gốc 235.795 dòng) để xem chính các dòng trùng. Tệp `SQL/03_queries.sql` chứa lại 19 truy vấn dưới đây để chạy trong công cụ SQL bất kỳ hỗ trợ cú pháp DuckDB.

Ô dưới tạo cơ sở dữ liệu `Outputs/phishing.duckdb`, nạp ba bảng (`raw_urls` từ tệp gốc, `urls` từ `phishing_model_table.parquet`, `tld_codes` từ `tld_country_codes.csv`) và định nghĩa hàm `run` để chạy một truy vấn rồi lưu kết quả ra CSV.
Cách đọc: dòng in ra là số dòng của từng bảng. Bảng `raw_urls` phải có 235.795 dòng, bảng `urls` phải có 235.370 dòng và 89 cột như Bước 2 đã lưu.

In [1]:
import os
import json
import duckdb
import numpy as np
import pandas as pd
DATA = "../Data/"
OUT = "../Outputs/"
SQL_DIR = "../SQL/"
os.makedirs(SQL_DIR, exist_ok=True)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda v: f"{v:.6g}")
DB_PATH = OUT + "phishing.duckdb"
if "con" in globals():
    con.close()                                                    # đóng kết nối cũ nếu chạy lại ô này, Windows không cho xoá tệp đang mở
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)                                             # tạo lại từ đầu để mỗi lần chạy cho cùng kết quả
con = duckdb.connect(DB_PATH)
con.execute("SET enable_progress_bar = false")                    # tắt thanh tiến trình để kết quả in ra gọn

raw = pd.read_csv(DATA + "PhiUSIIL_Phishing_URL_Dataset.csv", encoding="utf-8-sig")
raw["phishing"] = 1 - raw["label"]                                # 1 là lừa đảo, như ở Bước 2
con.register("raw_df", raw)
con.execute("CREATE TABLE raw_urls AS SELECT * FROM raw_df")      # tệp gốc 235.795 dòng, dùng cho truy vấn về dòng trùng và tên tệp
con.unregister("raw_df")
con.execute(f"CREATE TABLE urls AS SELECT * FROM read_parquet('{OUT}phishing_model_table.parquet')")    # bảng mô hình của Bước 2
con.execute(f"CREATE TABLE tld_codes AS SELECT * FROM read_csv('{OUT}tld_country_codes.csv', header = true)")
groups = json.load(open(OUT + "feature_groups.json", encoding="utf-8"))
QUERIES = {}                                                       # gom các truy vấn để ghi ra SQL/03_queries.sql


def run(name, title, sql):
    """Chạy một truy vấn, lưu kết quả ra CSV và trả về bảng kết quả."""
    QUERIES[name] = (title, sql.strip())
    result = con.execute(sql).df()
    result.to_csv(OUT + f"{name}.csv", index=False)
    return result


print("Đã nạp:", {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ["raw_urls", "urls", "tld_codes"]})
print("Số cột của bảng urls:", len(con.execute("DESCRIBE urls").df()), "| phiên bản DuckDB:", duckdb.__version__)
df = pd.read_parquet(OUT + "phishing_model_table.parquet")         # bảng pandas để đối chiếu với kết quả SQL

Đã nạp: {'raw_urls': 235795, 'urls': 235370, 'tld_codes': 253}
Số cột của bảng urls: 89 | phiên bản DuckDB: 1.5.5


## 3.1 Dữ liệu có đủ và đúng kích thước không
Ô dưới đếm số địa chỉ, số URL, số tên miền, số tên miền đăng ký và số TLD của từng lớp. `GROUP BY ROLLUP` thêm một dòng tổng cho cả bảng, và hàm `GROUPING` cho biết dòng nào là dòng tổng.
Cách đọc: số URL khác nhau phải bằng số địa chỉ (đã bỏ trùng ở Bước 2). Dòng tổng của số tên miền nhỏ hơn tổng hai lớp nếu có tên miền chung.

In [2]:
q1 = run("sql_01_overview", "Tổng quan bảng urls theo lớp, kèm một dòng tổng", """
SELECT CASE WHEN GROUPING(phishing) = 1 THEN 'all' WHEN phishing = 1 THEN 'phishing' ELSE 'legitimate' END AS lop,
       COUNT(*)                     AS so_dia_chi,
       COUNT(DISTINCT URL)          AS so_url_khac_nhau,
       COUNT(DISTINCT Domain)       AS so_ten_mien,
       COUNT(DISTINCT reg_domain)   AS so_ten_mien_dang_ky,
       COUNT(DISTINCT tld_clean)    AS so_tld,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM urls), 2)  AS ty_trong_pct
FROM urls
GROUP BY ROLLUP (phishing)
ORDER BY GROUPING(phishing), phishing
""")
print(q1.to_string(index=False))
check = [len(df), df.URL.nunique(), df.Domain.nunique(), df.reg_domain.nunique(), df.tld_clean.nunique()]
print("\nĐối chiếu pandas (dòng all):", dict(zip(["so_dia_chi", "so_url_khac_nhau", "so_ten_mien", "so_ten_mien_dang_ky", "so_tld"], check)),
      "| khớp:", q1.iloc[-1, 1:6].astype(int).tolist() == check)

       lop  so_dia_chi  so_url_khac_nhau  so_ten_mien  so_ten_mien_dang_ky  so_tld  ty_trong_pct
legitimate      134850            134850       134850               130640     473         57.29
  phishing      100520            100520        85290                43491     439         42.71
       all      235370            235370       220086               173977     578           100

Đối chiếu pandas (dòng all): {'so_dia_chi': 235370, 'so_url_khac_nhau': 235370, 'so_ten_mien': 220086, 'so_ten_mien_dang_ky': 173977, 'so_tld': 578} | khớp: True


**Nhận xét:** bảng có đúng 235.370 địa chỉ và 235.370 URL khác nhau, nên không còn dòng trùng. Lớp hợp lệ có 134.850 địa chỉ trên 134.850 tên miền khác nhau, tức mỗi trang hợp lệ là một tên miền riêng. Lớp lừa đảo có 100.520 địa chỉ nhưng chỉ 43.491 tên miền đăng ký: trung bình một tên miền đăng ký chứa hơn hai địa chỉ lừa đảo.
Cả 578 TLD và 173.977 tên miền đăng ký khớp với pandas và với Bước 2.

## 3.2 Địa chỉ trùng thuộc lớp nào và khác nhau ở đâu
Ô dưới chạy trên bảng `raw_urls` (tệp gốc). Bảng tạm `dup` gom các URL xuất hiện hơn một lần bằng `HAVING COUNT(*) > 1`, rồi đếm số URL mang hai nhãn và số cặp có `URLLength` lệch nhau đúng 1.
Cách đọc: cột `so_dong_bi_bo` là số dòng Bước 2 đã bỏ. Cột cuối cho biết bao nhiêu cặp chỉ khác nhau ở cách tính độ dài.

In [3]:
q2 = run("sql_02_duplicates", "Địa chỉ xuất hiện hơn một lần trong tệp gốc, theo lớp", """
WITH dup AS (
    SELECT URL,
           COUNT(*)                   AS so_lan,
           COUNT(DISTINCT phishing)   AS so_nhan,
           MIN(phishing)              AS nhan,
           MAX(URLLength) - MIN(URLLength) AS lech_URLLength
    FROM raw_urls
    GROUP BY URL
    HAVING COUNT(*) > 1
)
SELECT CASE nhan WHEN 1 THEN 'phishing' ELSE 'legitimate' END AS lop,
       COUNT(*)                         AS so_url_trung,
       SUM(so_lan)                      AS so_dong,
       SUM(so_lan - 1)                  AS so_dong_bi_bo,
       SUM(CASE WHEN so_nhan > 1 THEN 1 ELSE 0 END) AS url_mang_hai_nhan,
       SUM(CASE WHEN lech_URLLength = 1 THEN 1 ELSE 0 END) AS cap_lech_URLLength_1
FROM dup
GROUP BY lop
""")
print(q2.to_string(index=False))
pandas_dup = raw[raw.URL.duplicated(keep=False)]
print("\nĐối chiếu pandas: URL trùng", pandas_dup.URL.nunique(), "| dòng bị bỏ", int(raw.URL.duplicated().sum()), "| tệp gốc trừ dòng bị bỏ:", len(raw) - int(raw.URL.duplicated().sum()), "= số dòng bảng urls", len(df))

     lop  so_url_trung  so_dong  so_dong_bi_bo  url_mang_hai_nhan  cap_lech_URLLength_1
phishing           425      850            425                  0                   425

Đối chiếu pandas: URL trùng 425 | dòng bị bỏ 425 | tệp gốc trừ dòng bị bỏ: 235370 = số dòng bảng urls 235370


**Nhận xét:** cả 425 URL trùng (850 dòng) đều thuộc lớp lừa đảo, không có URL nào mang hai nhãn, và 425 dòng bị bỏ khớp với Bước 2: 235.795 trừ 425 bằng 235.370. Cả 425 cặp đều có `URLLength` lệch nhau đúng 1, xác nhận bằng SQL điều Bước 2 (mục 2.3) đã thấy bằng pandas: cùng một địa chỉ đã đi qua hai cách xử lý khác nhau của tác giả.

## 3.3 Hai lớp đặt cạnh nhau
Ô dưới so hai lớp trên chín chỉ số: dạng tiền tố, cờ HTTPS, có đường dẫn, độ dài thật, chỉ số tương đồng, số dòng mã và tỷ lệ trang không có ảnh. `AVG` của một điều kiện (`CASE WHEN ... THEN 1 ELSE 0 END`) cho ra tỷ lệ phần trăm.
Cách đọc: mỗi hàng một chỉ số, hai cột là hai lớp. Chỉ số nào cách xa nhau là chỉ số tách hai lớp.

In [4]:
q3 = run("sql_03_two_classes", "Hai lớp đặt cạnh nhau: dạng địa chỉ, chỉ số tương đồng và nội dung trang", """
SELECT CASE phishing WHEN 1 THEN 'phishing' ELSE 'legitimate' END       AS lop,
       COUNT(*)                                                         AS so_dia_chi,
       ROUND(100.0 * AVG(starts_https_www), 2)                          AS bat_dau_https_www_pct,
       ROUND(100.0 * AVG(IsHTTPS), 2)                                   AS https_pct,
       ROUND(100.0 * AVG(has_path), 2)                                  AS co_duong_dan_pct,
       MEDIAN(url_length_true)                                          AS do_dai_that_trung_vi,
       ROUND(AVG(URLSimilarityIndex), 2)                                AS chi_so_tuong_dong_tb,
       ROUND(100.0 * AVG(CASE WHEN URLSimilarityIndex = 100 THEN 1 ELSE 0 END), 2) AS chi_so_bang_100_pct,
       MEDIAN(LineOfCode)                                               AS dong_ma_trung_vi,
       ROUND(100.0 * AVG(CASE WHEN NoOfImage = 0 THEN 1 ELSE 0 END), 1)  AS khong_anh_pct
FROM urls
GROUP BY phishing
ORDER BY phishing
""")
print(q3.T.to_string(header=False))
p = df.groupby("phishing").agg(www=("starts_https_www", "mean"), path=("has_path", "mean"), med=("url_length_true", "median"), loc=("LineOfCode", "median"))
print("\nĐối chiếu pandas: bắt đầu https://www. (%)", (100 * p.www).round(2).tolist(), "| có đường dẫn (%)", (100 * p.path).round(2).tolist(),
      "| độ dài thật trung vị", p.med.tolist(), "| dòng mã trung vị", p["loc"].tolist())

lop                    legitimate  phishing
so_dia_chi                 134850    100520
bat_dau_https_www_pct         100      2.43
https_pct                     100     49.12
co_duong_dan_pct                0      59.7
do_dai_that_trung_vi           27        34
chi_so_tuong_dong_tb          100     49.65
chi_so_bang_100_pct           100      0.78
dong_ma_trung_vi             1105        12
khong_anh_pct                   1      83.5

Đối chiếu pandas: bắt đầu https://www. (%) [100.0, 2.43] | có đường dẫn (%) [0.0, 59.7] | độ dài thật trung vị [27.0, 34.0] | dòng mã trung vị [1105.0, 12.0]


**Nhận xét:** năm chỉ số gần như tách trọn hai lớp: bắt đầu bằng `https://www.` (100% so với 2,43%), có đường dẫn (0% so với 59,7%), chỉ số tương đồng bằng 100 (100% so với 0,78%), trung vị dòng mã (1.105 so với 12) và trang không có ảnh (1% so với 83,5%). Chỉ có độ dài thật là chồng lấn nhiều (trung vị 27 so với 34 ký tự).
Năm chỉ số này tách hai lớp chủ yếu vì cách thu thập chứ không vì hành vi lừa đảo: ngoài đời trang hợp lệ cũng có đường dẫn (`/tin-tuc/...`), cũng dùng `http://` hay không có `www.`, nhưng trong bộ dữ liệu này lớp hợp lệ chỉ gồm trang chủ dạng `https://www.<tên miền>`, còn trang lừa đảo phần lớn được lưu khi đã trống.

## 3.4 Bốn dạng tiền tố của địa chỉ
Ô dưới chia mọi địa chỉ thành bốn dạng tiền tố bằng `LIKE` và `CASE`, rồi đếm theo lớp bằng `COUNT(*) FILTER (WHERE ...)`, một cách xoay bảng (pivot) ngay trong SQL.
Cách đọc: mỗi hàng một dạng tiền tố. Hàng nào có cột `legitimate` bằng 0 thì mọi địa chỉ có tiền tố ấy đều là lừa đảo.

In [5]:
q4 = run("sql_04_prefix_shapes", "Bốn dạng tiền tố của địa chỉ, đếm theo lớp", """
WITH shaped AS (
    SELECT phishing,
           CASE WHEN URL LIKE 'https://www.%' THEN '1 https://www.'
                WHEN URL LIKE 'https://%'     THEN '2 https:// (no www)'
                WHEN URL LIKE 'http://www.%'  THEN '3 http://www.'
                ELSE                               '4 http:// (no www)' END AS tien_to
    FROM urls
)
SELECT tien_to,
       COUNT(*) FILTER (WHERE phishing = 0)                                  AS legitimate,
       COUNT(*) FILTER (WHERE phishing = 1)                                  AS phishing,
       ROUND(100.0 * COUNT(*) FILTER (WHERE phishing = 1) / COUNT(*), 2)     AS lua_dao_pct
FROM shaped
GROUP BY tien_to
ORDER BY tien_to
""")
print(q4.to_string(index=False))
shape = np.select([df.URL.str.startswith("https://www."), df.URL.str.startswith("https://"), df.URL.str.startswith("http://www.")], ["1", "2", "3"], "4")
print("\nĐối chiếu pandas:", pd.crosstab(shape, df.phishing).values.tolist(), "| khớp:", pd.crosstab(shape, df.phishing).values.tolist() == q4[["legitimate", "phishing"]].values.tolist())

            tien_to  legitimate  phishing  lua_dao_pct
     1 https://www.      134850      2445         1.78
2 https:// (no www)           0     46440          100
      3 http://www.           0     39275          100
 4 http:// (no www)           0     12360          100

Đối chiếu pandas: [[134850, 2445], [0, 46440], [0, 39275], [0, 12360]] | khớp: True


**Nhận xét:** cả 134.850 địa chỉ hợp lệ đều ở dạng `https://www.`, ba dạng còn lại (`https://` không có `www.`, `http://www.`, `http://`) chỉ có địa chỉ lừa đảo, tổng cộng 98.075 địa chỉ. Dạng `https://www.` có 2.445 địa chỉ lừa đảo, chỉ chiếm 1,78% dạng ấy.
Nói cách khác, chỉ cần nhìn tiền tố là phân loại đúng 98.075 + 134.850 = 232.925 trên 235.370 địa chỉ. Đây là gốc của quy tắc một dòng.

## 3.5 Bốn quy tắc viết tay
Ô dưới chấm bốn quy tắc trên cả bảng: chỉ số tương đồng dưới 100, không bắt đầu bằng `https://www.`, có đường dẫn, và gộp hai quy tắc giữa. `UNION ALL` xếp chồng bốn quy tắc thành một bảng dài để một câu `GROUP BY` chấm cả bốn.
Cách đọc: độ chính xác là tỷ lệ đoán đúng trên mọi địa chỉ, tỷ lệ bắt được chỉ tính trên địa chỉ lừa đảo, báo động giả chỉ tính trên địa chỉ hợp lệ. Cột cuối là số địa chỉ hợp lệ bị chặn nhầm.

In [6]:
q5 = run("sql_05_one_line_rules", "Bốn quy tắc viết tay trên toàn bộ 235.370 địa chỉ: độ chính xác, tỷ lệ bắt được và báo động giả", """
WITH flags AS (
    SELECT phishing,
           CASE WHEN URLSimilarityIndex < 100 THEN 1 ELSE 0 END                AS r_similarity,
           1 - starts_https_www                                                AS r_prefix,
           has_path                                                            AS r_path,
           CASE WHEN starts_https_www = 0 OR has_path = 1 THEN 1 ELSE 0 END    AS r_prefix_or_path
    FROM urls
),
long AS (
    SELECT '1 similarity index < 100' AS quy_tac, phishing, r_similarity AS flag FROM flags
    UNION ALL SELECT '2 not https://www.',          phishing, r_prefix         FROM flags
    UNION ALL SELECT '3 has a path',                phishing, r_path           FROM flags
    UNION ALL SELECT '4 not https://www. or path',  phishing, r_prefix_or_path FROM flags
)
SELECT quy_tac,
       ROUND(100.0 * AVG(CASE WHEN flag = phishing THEN 1 ELSE 0 END), 2)  AS do_chinh_xac_pct,
       ROUND(100.0 * AVG(flag) FILTER (WHERE phishing = 1), 2)             AS bat_duoc_pct,
       ROUND(100.0 * AVG(flag) FILTER (WHERE phishing = 0), 2)             AS bao_dong_gia_pct,
       SUM(flag) FILTER (WHERE phishing = 0)                               AS so_dia_chi_hop_le_bi_chan
FROM long
GROUP BY quy_tac
ORDER BY quy_tac
""")
print(q5.to_string(index=False))
y = df.phishing
for name, flag in [("prefix", 1 - df.starts_https_www), ("prefix_or_path", ((df.starts_https_www == 0) | (df.has_path == 1)).astype(int))]:
    print("Đối chiếu pandas,", name, ": độ chính xác %.2f%%, bắt %.2f%%, báo động giả %.2f%%" % (100 * (flag == y).mean(), 100 * flag[y == 1].mean(), 100 * flag[y == 0].mean()))

                   quy_tac  do_chinh_xac_pct  bat_duoc_pct  bao_dong_gia_pct  so_dia_chi_hop_le_bi_chan
  1 similarity index < 100             99.67         99.22                 0                          0
        2 not https://www.             98.96         97.57                 0                          0
              3 has a path             82.79          59.7                 0                          0
4 not https://www. or path             99.73         99.38                 0                          0
Đối chiếu pandas, prefix : độ chính xác 98.96%, bắt 97.57%, báo động giả 0.00%
Đối chiếu pandas, prefix_or_path : độ chính xác 99.73%, bắt 99.38%, báo động giả 0.00%


**Nhận xét:** cả bốn quy tắc đều không chặn nhầm địa chỉ hợp lệ nào. Quy tắc tiền tố đạt 98,96% độ chính xác và bắt 97,57% địa chỉ lừa đảo. Quy tắc gộp "không bắt đầu bằng `https://www.` hoặc có đường dẫn" đạt 99,73% và bắt 99,38%, cao hơn cả quy tắc dựa trên chỉ số tương đồng của tác giả (99,67%). Kết quả khớp với pandas và với Bước 1.
Đây là mốc mà mười mô hình ở Bước 6 phải vượt, và vượt ở mức báo động giả bằng 0.

## 3.6 Chỉ số tương đồng theo khoảng 10 điểm
Ô dưới chia chỉ số tương đồng thành các khoảng 10 điểm, đếm số địa chỉ, số địa chỉ hợp lệ, tỷ lệ lừa đảo và phần trăm tích luỹ bằng `SUM(COUNT(*)) OVER (ORDER BY ...)`.
Cách đọc: cột `hop_le` cho biết khoảng nào có địa chỉ hợp lệ. Cột tích luỹ cho biết bao nhiêu phần trăm địa chỉ có chỉ số nhỏ hơn hoặc bằng khoảng ấy.

In [7]:
q6 = run("sql_06_similarity_bands", "Chỉ số tương đồng theo khoảng 10 điểm: số địa chỉ, tỷ lệ lừa đảo và phần trăm tích luỹ", """
SELECT LEAST(CAST(FLOOR(URLSimilarityIndex / 10) * 10 AS INTEGER), 100)            AS tu_diem,
       COUNT(*)                                                                    AS so_dia_chi,
       COUNT(*) FILTER (WHERE phishing = 0)                                        AS hop_le,
       ROUND(100.0 * AVG(phishing), 2)                                             AS lua_dao_pct,
       ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY tu_diem) / SUM(COUNT(*)) OVER (), 2) AS tich_luy_pct
FROM urls
GROUP BY tu_diem
ORDER BY tu_diem
""")
print(q6.to_string(index=False))
band = np.minimum(np.floor(df.URLSimilarityIndex / 10) * 10, 100)
print("\nĐối chiếu pandas: địa chỉ có chỉ số dưới 100:", int((df.URLSimilarityIndex < 100).sum()), "| trong đó hợp lệ:", int(((df.URLSimilarityIndex < 100) & (df.phishing == 0)).sum()),
      "| số địa chỉ ở khoảng 100:", int((band == 100).sum()))

 tu_diem  so_dia_chi  hop_le  lua_dao_pct  tich_luy_pct
       0        4002       0          100           1.7
      10        8584       0          100          5.35
      20       10274       0          100          9.71
      30       11863       0          100         14.75
      40       13515       0          100         20.49
      50       14727       0          100         26.75
      60       16824       0          100          33.9
      70       12611       0          100         39.26
      80        6254       0          100         41.91
      90        1080       0          100         42.37
     100      135636  134850         0.58           100

Đối chiếu pandas: địa chỉ có chỉ số dưới 100: 99734 | trong đó hợp lệ: 0 | số địa chỉ ở khoảng 100: 135636


**Nhận xét:** mười khoảng từ 0 tới 99 điểm không có địa chỉ hợp lệ nào, cộng lại 99.734 địa chỉ, đều là lừa đảo. Khoảng 100 gồm toàn bộ 134.850 địa chỉ hợp lệ và chỉ 786 địa chỉ lừa đảo (0,58%).
Chỉ số tương đồng vì vậy không phải một thang đo liên tục mà là một công tắc: bằng 100 là gần như chắc hợp lệ, dưới 100 là chắc chắn lừa đảo. Điều này khớp với cách tác giả dựng chỉ số bằng cách so với danh sách địa chỉ hợp lệ.

## 3.7 Mười lăm TLD đông nhất
Ô dưới đếm số địa chỉ, tỷ lệ lừa đảo và tỷ lệ trang chủ của từng TLD, xếp hạng bằng `RANK() OVER`, tính tỷ trọng tích luỹ và giữ 15 hạng đầu bằng `QUALIFY`, một mệnh đề lọc trên kết quả của hàm cửa sổ.
Cách đọc: cột tích luỹ cho biết 15 TLD đầu giữ bao nhiêu phần trăm dữ liệu. Hai cột cuối cho thấy TLD nào gần như chỉ có một lớp.

In [8]:
q7 = run("sql_07_tld_rank", "Mười lăm TLD đông nhất: xếp hạng theo số địa chỉ, tỷ trọng và tỷ lệ lừa đảo", """
WITH by_tld AS (
    SELECT tld_clean,
           COUNT(*)                          AS so_dia_chi,
           ROUND(100.0 * AVG(phishing), 1)   AS lua_dao_pct,
           ROUND(100.0 * AVG(homepage_only), 1) AS trang_chu_pct
    FROM urls
    GROUP BY tld_clean
)
SELECT RANK() OVER (ORDER BY so_dia_chi DESC)                                  AS hang,
       tld_clean, so_dia_chi,
       ROUND(100.0 * so_dia_chi / SUM(so_dia_chi) OVER (), 2)                  AS ty_trong_pct,
       ROUND(100.0 * SUM(so_dia_chi) OVER (ORDER BY so_dia_chi DESC) / SUM(so_dia_chi) OVER (), 1) AS tich_luy_pct,
       lua_dao_pct, trang_chu_pct
FROM by_tld
QUALIFY hang <= 15
ORDER BY hang
""")
print(q7.to_string(index=False))
top = df.tld_clean.value_counts().head(15)
print("\nĐối chiếu pandas (15 TLD đông nhất):", top.index.tolist() == q7.tld_clean.tolist(), "| số địa chỉ khớp:", top.tolist() == q7.so_dia_chi.tolist())

 hang tld_clean  so_dia_chi  ty_trong_pct  tich_luy_pct  lua_dao_pct  trang_chu_pct
    1       com      112380         47.75          47.7         38.8           78.1
    2       org       18792          7.98          55.7         12.1           96.3
    3       net        7048          2.99          58.7         43.3           81.2
    4       app        6467          2.75          61.5         97.8              6
    5        uk        6395          2.72          64.2            5           98.1
    6        co        5408           2.3          66.5         91.5           35.5
    7        io        4174          1.77          68.3         89.7           16.7
    8        de        3994           1.7            70         17.1           91.2
    9        ru        3870          1.64          71.6           77           67.1
   10        au        2979          1.27          72.9         12.5           91.8
   11       top        2327          0.99          73.9         99.9        

**Nhận xét:** ba TLD đầu giữ 58,7% dữ liệu và 15 TLD đầu giữ 77,4%. Tỷ lệ lừa đảo chạy từ 0,3% (`.edu`) tới 99,9% (`.top`). Các TLD gần như toàn lừa đảo (`.app` 97,8%, `.dev` 98,6%, `.io` 89,7%) cũng có rất ít trang chủ (6%, 8,4%, 16,7%), còn các TLD gần như toàn hợp lệ (`.uk`, `.edu`, `.org`) có trên 96% trang chủ.
So với Bước 1, `.com` có 112.380 thay vì 112.382 địa chỉ và `.net` có 7.048 thay vì 7.076: Bước 2 đã đưa 30 tên máy chủ mà tác giả gắn cờ `IsDomainIP` (dạng `64.47.167.72.host.secureserver.net`) vào nhóm `(ip address)`.

## 3.8 Ba loại TLD, ghép với bảng mã nước
Ô dưới ghép bảng `urls` với bảng `tld_codes` bằng `LEFT JOIN` theo mã TLD, rồi so ba loại TLD: chung (`com`, `org`, `app`...), mã nước (`uk`, `de`, `vn`...) và địa chỉ IP.
Cách đọc: `LEFT JOIN` giữ mọi địa chỉ, kể cả địa chỉ không tìm được tên nước. Cột `so_nuoc` chỉ khác 0 ở loại mã nước.

In [9]:
q8 = run("sql_08_tld_type", "Ba loại TLD (chung, mã nước, địa chỉ IP) ghép với bảng mã nước", """
SELECT u.tld_type                                             AS loai_tld,
       COUNT(*)                                               AS so_dia_chi,
       COUNT(DISTINCT u.tld_clean)                            AS so_tld,
       COUNT(DISTINCT c.country)                              AS so_nuoc,
       ROUND(100.0 * AVG(u.phishing), 2)                      AS lua_dao_pct,
       ROUND(100.0 * AVG(u.homepage_only), 1)                 AS trang_chu_pct
FROM urls u
LEFT JOIN tld_codes c ON c.code = u.tld_clean
GROUP BY u.tld_type
ORDER BY so_dia_chi DESC
""")
print(q8.to_string(index=False))
print("\nĐối chiếu pandas:", df.tld_type.value_counts().to_dict(), "| số nước:", df.loc[df.tld_type == "country code", "country"].nunique())

    loai_tld  so_dia_chi  so_tld  so_nuoc  lua_dao_pct  trang_chu_pct
     generic      165858     346        0        42.76           74.1
country code       68876     231      231        42.05           76.1
  ip address         636       1        0          100            1.4

Đối chiếu pandas: {'generic': 165858, 'country code': 68876, 'ip address': 636} | số nước: 231


**Nhận xét:** TLD chung chiếm 165.858 địa chỉ trên 346 TLD, TLD mã nước chiếm 68.876 địa chỉ trên 231 TLD ứng với 231 nước và vùng lãnh thổ, nhóm địa chỉ IP có 636 địa chỉ. Tỷ lệ lừa đảo của TLD chung và TLD mã nước gần như bằng nhau (42,76% và 42,05%), tỷ lệ trang chủ cũng gần nhau (74,1% và 76,1%), nên loại TLD tự nó không tách hai lớp.
Nhóm địa chỉ IP toàn là lừa đảo và chỉ 1,4% là trang chủ. Lưu ý: 30 trong 636 địa chỉ của nhóm này là tên máy chủ có chứa địa chỉ IP (mục 3.7), vì nhóm dùng cờ `IsDomainIP` của tác giả.

## 3.9 Địa lý theo TLD mã nước
Ô dưới giữ các TLD mã nước có từ 500 địa chỉ (`HAVING COUNT(*) >= 500`), xếp hạng theo tỷ lệ lừa đảo bằng `DENSE_RANK` và chia thành ba nhóm bằng nhau bằng `NTILE(3)`.
Cách đọc: hạng 1 là TLD có tỷ lệ lừa đảo cao nhất. Đây là nơi đăng ký tên miền, không phải nơi đặt máy chủ hay nơi người lập trang sống (xem lưu ý ở Bước 2, mục 2.8).

In [10]:
q9 = run("sql_09_countries", "Địa lý theo TLD mã nước: các nước có từ 500 địa chỉ, xếp hạng theo tỷ lệ lừa đảo", """
WITH by_country AS (
    SELECT c.country, u.tld_clean,
           COUNT(*)                          AS so_dia_chi,
           ROUND(100.0 * AVG(u.phishing), 1) AS lua_dao_pct
    FROM urls u
    JOIN tld_codes c ON c.code = u.tld_clean
    GROUP BY c.country, u.tld_clean
    HAVING COUNT(*) >= 500
)
SELECT DENSE_RANK() OVER (ORDER BY lua_dao_pct DESC)  AS hang_lua_dao,
       country, tld_clean, so_dia_chi, lua_dao_pct,
       NTILE(3) OVER (ORDER BY lua_dao_pct DESC)      AS nhom_ba
FROM by_country
ORDER BY hang_lua_dao, so_dia_chi DESC
""")
print(q9.to_string(index=False))
cc = df[df.tld_type == "country code"].groupby("tld_clean").agg(n=("phishing", "size"), p=("phishing", "mean"))
print("\nĐối chiếu pandas: số TLD mã nước có từ 500 địa chỉ:", int((cc.n >= 500).sum()), "| khớp:", int((cc.n >= 500).sum()) == len(q9))

 hang_lua_dao                        country tld_clean  so_dia_chi  lua_dao_pct  nhom_ba
            1       Central African Republic        cf        1203          100        1
            2                           Mali        ml         994         99.9        1
            3                          Gabon        ga        1111         99.6        1
            4                       Colombia        co        5408         91.5        1
            5 British Indian Ocean Territory        io        4174         89.7        1
            6                     Montenegro        me         998         84.7        1
            7             Russian Federation        ru        3870           77        1
            8                      Indonesia        id         801         69.8        1
            9                          China        cn         992         69.5        1
           10                         Poland        pl        1329         59.5        1
           11        

**Nhận xét:** 32 TLD mã nước có từ 500 địa chỉ. Ba TLD đầu gần như toàn lừa đảo: `.cf` của Cộng hoà Trung Phi (100%), `.ml` của Mali (99,9%) và `.ga` của Gabon (99,6%). Cả ba từng được cấp miễn phí, nên bị dùng nhiều để dựng trang lừa đảo. Kế đó là `.co` và `.io`, hai mã nước được bán như TLD chung.
Nhóm thấp nhất gồm các nước có cơ quan đăng ký chặt chẽ: `.uk` 5,0%, `.jp` 6,2%, `.ca` 6,9%. Khác biệt giữa các TLD rất lớn, nên góc nhìn địa lý giải thích được vì sao phép thử TLD chưa thấy khó, nhưng không cho kết luận gì về nước nào lừa đảo nhiều hơn.

## 3.10 Độ tập trung của lừa đảo theo tên miền đăng ký
Ô dưới đếm địa chỉ lừa đảo và hợp lệ của từng tên miền đăng ký, đánh số bằng `ROW_NUMBER`, tính phần trăm tích luỹ trên tổng số địa chỉ lừa đảo (lấy bằng một truy vấn con) và giữ 20 tên miền đầu.
Cách đọc: cột cuối cho biết 20 tên miền đầu giữ bao nhiêu phần trăm toàn bộ địa chỉ lừa đảo.

In [11]:
q10 = run("sql_10_registered_domains", "Độ tập trung của địa chỉ lừa đảo theo tên miền đăng ký: hai mươi tên miền đông nhất", """
WITH by_domain AS (
    SELECT reg_domain,
           COUNT(*)                            AS so_dia_chi,
           SUM(phishing)                       AS lua_dao,
           COUNT(*) - SUM(phishing)            AS hop_le
    FROM urls
    GROUP BY reg_domain
)
SELECT ROW_NUMBER() OVER (ORDER BY lua_dao DESC, reg_domain)                       AS hang,
       reg_domain, so_dia_chi, lua_dao, hop_le,
       ROUND(100.0 * SUM(lua_dao) OVER (ORDER BY lua_dao DESC, reg_domain) / (SELECT SUM(phishing) FROM urls), 2) AS tich_luy_lua_dao_pct
FROM by_domain
QUALIFY hang <= 20
ORDER BY hang
""")
print(q10.to_string(index=False))
ph = df[df.phishing == 1].reg_domain.value_counts()
print("\nĐối chiếu pandas: hai mươi tên miền đông nhất giữ %.2f%% địa chỉ lừa đảo | số tên miền chỉ có địa chỉ lừa đảo và có từ 100 địa chỉ: %d"
      % (100 * ph.head(20).sum() / ph.sum(), int(((df.groupby("reg_domain").phishing.agg(["size", "mean"]).query("size >= 100 and mean == 1")).shape[0]))))

 hang       reg_domain  so_dia_chi  lua_dao  hop_le  tich_luy_lua_dao_pct
    1          web.app        5718     5718       0                  5.69
    2  firebaseapp.com        5546     5546       0                 11.21
    3          repl.co        3746     3746       0                 14.93
    4   weeblysite.com        3079     3079       0                    18
    5          ipfs.io        1552     1552       0                 19.54
    6      workers.dev        1423     1423       0                 20.96
    7      square.site        1187     1187       0                 22.14
    8        dweb.link         959      959       0                 23.09
    9          xsph.ru         928      928       0                 24.01
   10  pantheonsite.io         891      891       0                  24.9
   11       google.com         782      782       0                 25.68
   12 godaddysites.com         770      770       0                 26.44
   13      duckdns.org         642    

**Nhận xét:** chỉ 20 tên miền đăng ký giữ 30,53% toàn bộ địa chỉ lừa đảo, và 19 trong 20 tên miền ấy không có địa chỉ hợp lệ nào. Gần như cả danh sách là dịch vụ lưu trữ miễn phí (`web.app`, `firebaseapp.com`, `repl.co`, `weeblysite.com`, `ipfs.io`, `workers.dev`), cộng thêm `google.com` (782 địa chỉ, có lẽ là trang Google Sites hoặc Forms).
Có 62 tên miền có từ 100 địa chỉ trở lên mà toàn là lừa đảo. Đây là lý do Bước 2 phải có cách chia theo tên miền: chia ngẫu nhiên thì các tên miền này nằm ở cả hai tập.

## 3.11 Phân bố độ dài địa chỉ sau khi cắt tiền tố
Ô dưới chia `pb_length` (độ dài địa chỉ sau khi cắt `http://`, `https://` và `www.`, tính ở Bước 2) thành bảy khoảng, đếm theo lớp, rồi tính tỷ lệ trong từng lớp bằng `SUM(...) OVER ()`.
Cách đọc: hai cột phần trăm, mỗi cột cộng lại 100%. So hai cột ở cùng một hàng để thấy khoảng nào lớp nào chiếm nhiều hơn.

In [12]:
q11 = run("sql_11_length_bands", "Phân bố độ dài địa chỉ sau khi cắt tiền tố (pb_length) theo khoảng, tỷ lệ trong từng lớp", """
WITH banded AS (
    SELECT phishing,
           CASE WHEN pb_length < 10 THEN '1 under 10'
                WHEN pb_length < 15 THEN '2 10-14'
                WHEN pb_length < 20 THEN '3 15-19'
                WHEN pb_length < 30 THEN '4 20-29'
                WHEN pb_length < 50 THEN '5 30-49'
                WHEN pb_length < 100 THEN '6 50-99'
                ELSE '7 100 or more' END AS khoang_do_dai
    FROM urls
)
SELECT khoang_do_dai,
       COUNT(*) FILTER (WHERE phishing = 0)                                                     AS hop_le,
       COUNT(*) FILTER (WHERE phishing = 1)                                                     AS lua_dao,
       ROUND(100.0 * COUNT(*) FILTER (WHERE phishing = 0) / SUM(COUNT(*) FILTER (WHERE phishing = 0)) OVER (), 2) AS hop_le_pct,
       ROUND(100.0 * COUNT(*) FILTER (WHERE phishing = 1) / SUM(COUNT(*) FILTER (WHERE phishing = 1)) OVER (), 2) AS lua_dao_pct
FROM banded
GROUP BY khoang_do_dai
ORDER BY khoang_do_dai
""")
print(q11.to_string(index=False))
bins = pd.cut(df.pb_length, [0, 10, 15, 20, 30, 50, 100, np.inf], right=False)
share = (100 * pd.crosstab(bins, df.phishing, normalize="columns")).round(2)
print("\nĐối chiếu pandas (hai cột phần trăm):", share.values.tolist() == q11[["hop_le_pct", "lua_dao_pct"]].values.tolist())

khoang_do_dai  hop_le  lua_dao  hop_le_pct  lua_dao_pct
   1 under 10   14658     3665       10.87         3.65
      2 10-14   49334    16325       36.58        16.24
      3 15-19   46326    16428       34.35        16.34
      4 20-29   23734    22901        17.6        22.78
      5 30-49     798    24119        0.59        23.99
      6 50-99       0    13122           0        13.05
7 100 or more       0     3960           0         3.94

Đối chiếu pandas (hai cột phần trăm): True


**Nhận xét:** 81,8% địa chỉ hợp lệ dài dưới 20 ký tự và không địa chỉ hợp lệ nào dài từ 50 ký tự trở lên. Lớp lừa đảo trải rộng hơn: 41% dài từ 30 ký tự trở lên, và 3.960 địa chỉ dài từ 100 ký tự trở lên.
Nhưng ở các khoảng ngắn hai lớp chồng lấn nhiều: 36.418 địa chỉ lừa đảo dài dưới 20 ký tự, tức có độ dài giống hệt một trang hợp lệ. Độ dài chỉ tách được phần đuôi, không tách được phần giữa.

## 3.12 Phân vị độ dài ở nhóm trang chủ
Ô dưới tính phân vị 5%, 25%, 50%, 75%, 95% và giá trị lớn nhất của `pb_length` cho từng lớp, tách địa chỉ là trang chủ khỏi địa chỉ có đường dẫn. `GROUP BY ALL` nhóm theo mọi cột không phải hàm tổng hợp.
Cách đọc: không có hàng "has a path" cho lớp hợp lệ, vì lớp hợp lệ không có địa chỉ nào có đường dẫn.

In [13]:
q12 = run("sql_12_length_quantiles", "Phân vị của pb_length theo lớp, trên toàn bộ và riêng các địa chỉ là trang chủ", """
SELECT CASE WHEN homepage_only = 1 THEN 'homepage only' ELSE 'has a path' END AS nhom,
       CASE phishing WHEN 1 THEN 'phishing' ELSE 'legitimate' END AS lop,
       COUNT(*)                              AS so_dia_chi,
       QUANTILE_CONT(pb_length, 0.05)        AS p05,
       QUANTILE_CONT(pb_length, 0.25)        AS p25,
       QUANTILE_CONT(pb_length, 0.50)        AS p50,
       QUANTILE_CONT(pb_length, 0.75)        AS p75,
       QUANTILE_CONT(pb_length, 0.95)        AS p95,
       MAX(pb_length)                        AS lon_nhat
FROM urls
GROUP BY ALL
ORDER BY nhom DESC, lop
""")
print(q12.to_string(index=False))
home = df[df.homepage_only == 1]
print("\nĐối chiếu pandas, trang chủ, trung vị theo lớp:", home.groupby("phishing").pb_length.median().to_dict(), "| p95:", home.groupby("phishing").pb_length.quantile(0.95).to_dict())

         nhom        lop  so_dia_chi  p05  p25  p50  p75  p95  lon_nhat
homepage only legitimate      134850    8   12   15   18   24        46
homepage only   phishing       40513    9   12   15   20   35       101
   has a path   phishing       60007   16   26   35   55  113      6089

Đối chiếu pandas, trang chủ, trung vị theo lớp: {0: 15.0, 1: 15.0} | p95: {0: 24.0, 1: 35.0}


**Nhận xét:** ở nhóm trang chủ, hai lớp có cùng trung vị 15 ký tự và cùng phân vị 25% là 12 ký tự. Chúng chỉ tách nhau ở đuôi: phân vị 95% là 24 ký tự ở lớp hợp lệ và 35 ký tự ở lớp lừa đảo.
Ngược lại, 60.007 địa chỉ lừa đảo có đường dẫn có trung vị 35 ký tự, gấp hơn hai lần. Phần lớn khác biệt về độ dài ở mục 3.11 vì vậy đến từ đường dẫn, mà đường dẫn lại là một lối tắt (không trang hợp lệ nào có).

## 3.13 Trang lừa đảo có thật sự trống không
Ô dưới đếm, cho từng lớp, tỷ lệ trang không có ảnh, không có JavaScript, không có CSS, không có cả ba, có dưới 50 dòng mã, có ô mật khẩu và có nút gửi.
Cách đọc: một trang đăng nhập giả còn hoạt động phải có ô mật khẩu và nút gửi. Nếu tỷ lệ ấy thấp ở lớp lừa đảo thì trang đã được lưu khi đã chết hoặc bị chặn.

In [14]:
q13 = run("sql_13_empty_pages", "Trang gần như trống và ô mật khẩu, theo lớp", """
SELECT CASE phishing WHEN 1 THEN 'phishing' ELSE 'legitimate' END                    AS lop,
       ROUND(100.0 * AVG(CASE WHEN NoOfImage = 0 THEN 1 ELSE 0 END), 1)               AS khong_anh_pct,
       ROUND(100.0 * AVG(CASE WHEN NoOfJS = 0 THEN 1 ELSE 0 END), 1)                  AS khong_js_pct,
       ROUND(100.0 * AVG(CASE WHEN NoOfCSS = 0 THEN 1 ELSE 0 END), 1)                 AS khong_css_pct,
       ROUND(100.0 * AVG(CASE WHEN NoOfImage = 0 AND NoOfJS = 0 AND NoOfCSS = 0 THEN 1 ELSE 0 END), 1) AS ca_ba_bang_0_pct,
       ROUND(100.0 * AVG(CASE WHEN LineOfCode < 50 THEN 1 ELSE 0 END), 1)             AS duoi_50_dong_ma_pct,
       ROUND(100.0 * AVG(HasPasswordField), 1)                                        AS co_o_mat_khau_pct,
       ROUND(100.0 * AVG(HasSubmitButton), 1)                                         AS co_nut_gui_pct
FROM urls
GROUP BY phishing
ORDER BY phishing
""")
print(q13.to_string(index=False))
empty = ((df.NoOfImage == 0) & (df.NoOfJS == 0) & (df.NoOfCSS == 0))
print("\nĐối chiếu pandas: cả ba bằng 0 (%):", (100 * empty.groupby(df.phishing).mean()).round(1).to_dict(), "| có ô mật khẩu (%):", (100 * df.groupby("phishing").HasPasswordField.mean()).round(1).to_dict())

       lop  khong_anh_pct  khong_js_pct  khong_css_pct  ca_ba_bang_0_pct  duoi_50_dong_ma_pct  co_o_mat_khau_pct  co_nut_gui_pct
legitimate              1           1.1              2                 0                    0               13.8            66.1
  phishing           83.5          77.2           80.4              68.7                 68.2                5.3             8.4

Đối chiếu pandas: cả ba bằng 0 (%): {0: 0.0, 1: 68.7} | có ô mật khẩu (%): {0: 13.8, 1: 5.3}


**Nhận xét:** 68,7% trang lừa đảo không có cả ảnh, JavaScript lẫn CSS, và 68,2% có dưới 50 dòng mã. Ở lớp hợp lệ hai con số ấy đều là 0%. Chỉ 5,3% trang lừa đảo có ô mật khẩu và 8,4% có nút gửi, trong khi trang hợp lệ có 13,8% và 66,1%.
Phần lớn trang lừa đảo trong bộ dữ liệu vì vậy là trang đã chết hoặc bị chặn lúc tải. Một bộ lọc học từ nhóm đặc trưng trang sẽ học "trang trống là lừa đảo", điều không giúp gì khi gặp một trang lừa đảo đang hoạt động.

## 3.14 Cột `URLLength` lệch theo nguồn nào
Ô dưới tính `URLLength` trừ độ dài thật của chuỗi `URL` bằng hàm `LENGTH`, rồi đếm số địa chỉ lệch -1, lệch 0 và lệch khác theo lớp và theo tiền tố tên tệp `source_prefix` (Bước 2, mục 2.6).
Cách đọc: lệch -1 là bị cắt mất một ký tự khi tác giả tính độ dài. Nếu tỷ lệ lệch -1 khác nhau giữa các tiền tố tên tệp thì lỗi này đi theo nguồn thu thập.

In [15]:
q14 = run("sql_14_length_gap", "URLLength trừ độ dài thật của chuỗi URL, theo lớp và theo tiền tố tên tệp", """
SELECT CASE phishing WHEN 1 THEN 'phishing' ELSE 'legitimate' END                 AS lop,
       source_prefix                                                               AS tien_to_ten_tep,
       COUNT(*)                                                                    AS so_dia_chi,
       COUNT(*) FILTER (WHERE URLLength - LENGTH(URL) = -1)                        AS lech_tru_1,
       COUNT(*) FILTER (WHERE URLLength - LENGTH(URL) = 0)                         AS lech_0,
       COUNT(*) FILTER (WHERE URLLength - LENGTH(URL) NOT IN (-1, 0))              AS lech_khac,
       ROUND(100.0 * COUNT(*) FILTER (WHERE URLLength - LENGTH(URL) = -1) / COUNT(*), 1) AS lech_tru_1_pct
FROM urls
GROUP BY ALL
ORDER BY lop, so_dia_chi DESC
""")
print(q14.to_string(index=False))
lech = df.URLLength - df.URL.str.len()
print("\nĐối chiếu pandas: lệch -1 theo lớp:", (lech == -1).groupby(df.phishing).sum().to_dict(), "| trong SQL:", q14.groupby("lop").lech_tru_1.sum().to_dict())

       lop tien_to_ten_tep  so_dia_chi  lech_tru_1  lech_0  lech_khac  lech_tru_1_pct
legitimate     digits only      134850      134850       0          0             100
  phishing     digits only       55021       21308   33711          2            38.7
  phishing              mw       36283       28610    7673          0            78.9
  phishing             oph        8288        1530    6758          0            18.5
  phishing              op         548         262     286          0            47.8
  phishing               m         380         380       0          0             100

Đối chiếu pandas: lệch -1 theo lớp: {0: 134850, 1: 52090} | trong SQL: {'legitimate': 134850, 'phishing': 52090}


**Nhận xét:** cả 134.850 địa chỉ hợp lệ đều lệch -1. Ở lớp lừa đảo, tỷ lệ lệch -1 khác nhau rất xa giữa các nguồn: 100% ở tên tệp bắt đầu bằng `m`, 78,9% ở `mw`, 47,8% ở `op`, 38,7% ở tên tệp chỉ gồm chữ số và 18,5% ở `oph`. Tổng cộng 52.090 địa chỉ lừa đảo lệch -1, khớp với pandas.
Lỗi cắt một ký tự vì vậy đi theo cách từng đợt dữ liệu được xử lý, không theo bản chất của địa chỉ. Mô hình dùng `URLLength` có thể học chính lỗi này.

## 3.15 Chia ngẫu nhiên để lộ tên miền bao nhiêu
Ô dưới dùng `UNPIVOT` để xoay mười cột chia của Bước 2 thành dạng dài (mỗi dòng một địa chỉ, một cách chia, một tập), lấy danh sách tên miền đăng ký của từng tập huấn luyện, rồi `LEFT JOIN` tập kiểm thử với danh sách ấy. Địa chỉ kiểm thử nào ghép được là địa chỉ có tên miền đã gặp lúc huấn luyện.
Cách đọc: cột `ten_mien_da_gap_pct` tính trên mọi địa chỉ kiểm thử, cột cuối chỉ tính trên địa chỉ lừa đảo.

In [16]:
q15 = run("sql_15_split_domain_overlap", "Địa chỉ kiểm thử có tên miền đăng ký đã gặp trong tập huấn luyện, hai cách chia, năm seed", """
WITH roles AS (
    UNPIVOT (SELECT row_id, reg_domain, phishing, split_s0, split_s1, split_s2, split_s3, split_s4,
                    gsplit_s0, gsplit_s1, gsplit_s2, gsplit_s3, gsplit_s4 FROM urls)
    ON split_s0, split_s1, split_s2, split_s3, split_s4, gsplit_s0, gsplit_s1, gsplit_s2, gsplit_s3, gsplit_s4
    INTO NAME cach_chia VALUE tap
),
train_domains AS (
    SELECT DISTINCT cach_chia, reg_domain FROM roles WHERE tap = 'train'
)
SELECT CASE WHEN r.cach_chia LIKE 'g%' THEN 'domain split' ELSE 'stratified' END   AS loai_chia,
       r.cach_chia,
       COUNT(*)                                                                    AS dia_chi_kiem_thu,
       ROUND(100.0 * AVG(r.phishing), 2)                                           AS lua_dao_pct,
       ROUND(100.0 * AVG(CASE WHEN t.reg_domain IS NOT NULL THEN 1 ELSE 0 END), 2) AS ten_mien_da_gap_pct,
       ROUND(100.0 * AVG(CASE WHEN t.reg_domain IS NOT NULL THEN 1 ELSE 0 END) FILTER (WHERE r.phishing = 1), 2) AS ten_mien_da_gap_lua_dao_pct
FROM roles r
LEFT JOIN train_domains t ON t.cach_chia = r.cach_chia AND t.reg_domain = r.reg_domain
WHERE r.tap = 'test'
GROUP BY ALL
ORDER BY loai_chia DESC, r.cach_chia
""")
print(q15.to_string(index=False))
seen = df.loc[df.split_s0 == "test", "reg_domain"].isin(set(df.loc[df.split_s0 == "train", "reg_domain"]))
print("\nĐối chiếu pandas, split_s0: %.2f%% | trung bình năm seed phân tầng trong SQL: %.2f%%" % (100 * seen.mean(), q15[q15.loai_chia == "stratified"].ten_mien_da_gap_pct.mean()))

   loai_chia cach_chia  dia_chi_kiem_thu  lua_dao_pct  ten_mien_da_gap_pct  ten_mien_da_gap_lua_dao_pct
  stratified  split_s0             47074        42.71                 27.3                        58.97
  stratified  split_s1             47074        42.71                27.48                        59.68
  stratified  split_s2             47074        42.71                27.33                        59.36
  stratified  split_s3             47074        42.71                27.18                        58.97
  stratified  split_s4             47074        42.71                27.66                        59.78
domain split gsplit_s0             43431        37.82                    0                            0
domain split gsplit_s1             56099        51.62                    0                            0
domain split gsplit_s2             49610        46.65                    0                            0
domain split gsplit_s3             41882        35.78           

**Nhận xét:** ở năm seed của cách chia phân tầng, 27,18% tới 27,66% địa chỉ kiểm thử có tên miền đã gặp lúc huấn luyện (trung bình 27,39%, khớp với Bước 2). Tính riêng địa chỉ lừa đảo thì con số này lên 58,97% tới 59,78%: cứ 10 địa chỉ lừa đảo trong tập kiểm thử thì gần 6 địa chỉ nằm trên một tên miền mô hình đã thấy.
Cách chia theo tên miền đưa cả hai con số về 0% ở mọi seed. Đổi lại, tập kiểm thử có kích thước (41.882 tới 56.099 dòng) và tỷ lệ lừa đảo (35,78% tới 51,62%) thay đổi theo seed.

## 3.16 Hai phần của phép thử TLD chưa thấy
Ô dưới so phần dùng để huấn luyện (`com`, `org`, `net`) với phần chỉ dùng để thử (các TLD khác) về kích thước, tỷ lệ lừa đảo, tỷ lệ trang chủ, dạng tiền tố và độ dài của từng lớp.
Cách đọc: nếu hai phần khác nhau nhiều thì mô hình huấn luyện trên phần đầu sẽ gặp một dữ liệu khác hẳn ở phần sau.

In [17]:
q16 = run("sql_16_tld_seen_unseen", "Hai phần của phép thử TLD chưa thấy: kích thước, tỷ lệ lừa đảo, dạng tiền tố và trang chủ", """
SELECT CASE tld_seen WHEN 1 THEN 'seen (com, org, net)' ELSE 'unseen (other TLDs)' END   AS phan,
       COUNT(*)                                                     AS so_dia_chi,
       COUNT(DISTINCT tld_clean)                                    AS so_tld,
       ROUND(100.0 * AVG(phishing), 2)                              AS lua_dao_pct,
       ROUND(100.0 * AVG(homepage_only), 1)                         AS trang_chu_pct,
       ROUND(100.0 * AVG(starts_https_www) FILTER (WHERE phishing = 1), 2) AS lua_dao_bat_dau_https_www_pct,
       ROUND(100.0 * AVG(IsHTTPS) FILTER (WHERE phishing = 1), 1)   AS lua_dao_https_pct,
       ROUND(AVG(pb_length) FILTER (WHERE phishing = 0), 1)         AS do_dai_tb_hop_le,
       ROUND(AVG(pb_length) FILTER (WHERE phishing = 1), 1)         AS do_dai_tb_lua_dao
FROM urls
GROUP BY tld_seen
ORDER BY tld_seen DESC
""")
print(q16.T.to_string(header=False))
print("\nĐối chiếu pandas:", df.groupby("tld_seen").size().to_dict(), (100 * df.groupby("tld_seen").phishing.mean()).round(2).to_dict())

phan                           seen (com, org, net)  unseen (other TLDs)
so_dia_chi                                   138220                97150
so_tld                                            3                  575
lua_dao_pct                                   35.39                53.12
trang_chu_pct                                  80.7                 65.7
lua_dao_bat_dau_https_www_pct                  2.56                 2.31
lua_dao_https_pct                              45.4                 52.7
do_dai_tb_hop_le                               15.8                 14.2
do_dai_tb_lua_dao                              35.8                 38.3

Đối chiếu pandas: {0: 97150, 1: 138220} {0: 53.12, 1: 35.39}


**Nhận xét:** phần huấn luyện có 138.220 địa chỉ trên 3 TLD với 35,39% lừa đảo, phần thử có 97.150 địa chỉ trên 575 TLD với 53,12% lừa đảo. Phần thử có ít trang chủ hơn (65,7% so với 80,7%), và địa chỉ lừa đảo ở phần thử dùng https nhiều hơn (52,7% so với 45,4%).
Nhưng tỷ lệ địa chỉ lừa đảo bắt đầu bằng `https://www.` gần như bằng nhau ở hai phần (2,56% và 2,31%). Lối tắt tiền tố vì vậy vẫn còn nguyên ở phần TLD chưa thấy, nên phép thử TLD chưa thấy chỉ khó với mô hình không dùng tiền tố.

## 3.17 Câu hỏi nghiên cứu 1: hai lớp ở nhóm trang chủ
Câu hỏi 1 hỏi hai lớp khác nhau thế nào về dạng địa chỉ, TLD và chỉ số tương đồng. Mục 3.3 đã trả lời trên toàn bộ dữ liệu. Ô dưới trả lời lại chỉ trên 175.363 địa chỉ là trang chủ, nhóm so hai lớp công bằng nhất vì không lớp nào có đường dẫn.
Cách đọc: ba hàng đầu là các lối tắt của tác giả, các hàng sau là đặc trưng không thấy tiền tố và loại TLD.

In [18]:
q17 = run("sql_17_rq1_homepage_classes", "Câu hỏi 1: hai lớp khác nhau thế nào khi chỉ xét các địa chỉ là trang chủ", """
SELECT CASE phishing WHEN 1 THEN 'phishing' ELSE 'legitimate' END                  AS lop,
       COUNT(*)                                                                    AS so_dia_chi,
       ROUND(100.0 * AVG(starts_https_www), 2)                                     AS bat_dau_https_www_pct,
       ROUND(100.0 * AVG(IsHTTPS), 2)                                              AS https_pct,
       ROUND(100.0 * AVG(CASE WHEN URLSimilarityIndex = 100 THEN 1 ELSE 0 END), 2) AS chi_so_bang_100_pct,
       MEDIAN(pb_length)                                                           AS do_dai_trung_vi,
       ROUND(AVG(pb_digits), 2)                                                    AS chu_so_tb,
       ROUND(100.0 * AVG(CASE WHEN pb_digits > 0 THEN 1 ELSE 0 END), 1)            AS co_chu_so_pct,
       ROUND(100.0 * AVG(CASE WHEN pb_hyphens > 0 THEN 1 ELSE 0 END), 1)           AS co_gach_ngang_pct,
       ROUND(100.0 * AVG(CASE WHEN tld_type = 'country code' THEN 1 ELSE 0 END), 1) AS tld_ma_nuoc_pct
FROM urls
WHERE homepage_only = 1
GROUP BY phishing
ORDER BY phishing
""")
print(q17.T.to_string(header=False))
print("\nĐối chiếu pandas: trang chủ theo lớp", home.groupby("phishing").size().to_dict(), "| bắt đầu https://www. (%):", (100 * home.groupby("phishing").starts_https_www.mean()).round(2).to_dict(),
      "| có chữ số (%):", (100 * (home.pb_digits > 0).groupby(home.phishing).mean()).round(1).to_dict())

lop                    legitimate  phishing
so_dia_chi                 134850     40513
bat_dau_https_www_pct         100      1.55
https_pct                     100      5.61
chi_so_bang_100_pct           100      1.87
do_dai_trung_vi                15        15
chu_so_tb                    0.05      1.03
co_chu_so_pct                 2.6      20.2
co_gach_ngang_pct             7.5        17
tld_ma_nuoc_pct              29.6      30.9

Đối chiếu pandas: trang chủ theo lớp {0: 134850, 1: 40513} | bắt đầu https://www. (%): {0: 100.0, 1: 1.55} | có chữ số (%): {0: 2.6, 1: 20.2}


**Nhận xét:** ở nhóm trang chủ, các lối tắt vẫn tách gần trọn hai lớp: 100% địa chỉ hợp lệ bắt đầu bằng `https://www.` so với 1,55% địa chỉ lừa đảo, 100% dùng https so với 5,61%, và 100% có chỉ số tương đồng bằng 100 so với 1,87%.
Còn các đặc trưng không thấy tiền tố thì gần nhau: cùng trung vị độ dài 15 ký tự, tỷ lệ TLD mã nước 29,6% so với 30,9%. Khác biệt rõ nhất là chữ số (20,2% địa chỉ lừa đảo có chữ số so với 2,6% địa chỉ hợp lệ) và gạch ngang (17,0% so với 7,5%).
**Trả lời câu hỏi 1:** hai lớp khác nhau gần như tuyệt đối ở tiền tố, đường dẫn và chỉ số tương đồng, những thứ phản ánh cách thu thập. Ở hình dạng của chính tên miền, chúng chỉ khác nhau một phần (chữ số, gạch ngang).

## 3.18 Câu hỏi nghiên cứu 2: mốc không cần huấn luyện ở năm seed
Câu hỏi 2 so mười mô hình ở Bước 6. SQL không huấn luyện mô hình, nhưng tính được mốc mà mô hình phải vượt. Ô dưới dùng `UNPIVOT` xoay năm cột chia phân tầng, chấm quy tắc tiền tố và quy tắc gộp trên đúng tập kiểm thử của từng seed, rồi thêm hàng trung bình và độ lệch chuẩn (`STDDEV_SAMP`).
Cách đọc: năm hàng đầu là năm seed, hai hàng cuối là trung bình và độ lệch chuẩn. Đây là đúng các tập kiểm thử mà Bước 6 dùng cho mười mô hình.

In [19]:
q18 = run("sql_18_rq2_rule_by_seed", "Câu hỏi 2: mốc không cần huấn luyện, quy tắc tiền tố trên tập kiểm thử của năm seed phân tầng", """
WITH roles AS (
    UNPIVOT (SELECT phishing, starts_https_www, has_path, split_s0, split_s1, split_s2, split_s3, split_s4 FROM urls)
    ON split_s0, split_s1, split_s2, split_s3, split_s4
    INTO NAME seed VALUE tap
),
scored AS (
    SELECT seed, phishing,
           1 - starts_https_www                                              AS r_prefix,
           CASE WHEN starts_https_www = 0 OR has_path = 1 THEN 1 ELSE 0 END  AS r_prefix_or_path
    FROM roles
    WHERE tap = 'test'
),
per_seed AS (
    SELECT seed, COUNT(*) AS dia_chi_kiem_thu,
           100.0 * AVG(CASE WHEN r_prefix = phishing THEN 1 ELSE 0 END)          AS acc_prefix,
           100.0 * AVG(r_prefix) FILTER (WHERE phishing = 1)                     AS caught_prefix,
           100.0 * AVG(r_prefix) FILTER (WHERE phishing = 0)                     AS fa_prefix,
           100.0 * AVG(CASE WHEN r_prefix_or_path = phishing THEN 1 ELSE 0 END)  AS acc_both,
           100.0 * AVG(r_prefix_or_path) FILTER (WHERE phishing = 1)             AS caught_both
    FROM scored
    GROUP BY seed
)
SELECT * FROM (
SELECT seed, dia_chi_kiem_thu,
       ROUND(acc_prefix, 2) AS do_chinh_xac_tien_to_pct, ROUND(caught_prefix, 2) AS bat_duoc_tien_to_pct, ROUND(fa_prefix, 2) AS bao_dong_gia_pct,
       ROUND(acc_both, 2) AS do_chinh_xac_tien_to_hoac_duong_dan_pct, ROUND(caught_both, 2) AS bat_duoc_tien_to_hoac_duong_dan_pct
FROM per_seed
UNION ALL
SELECT 'mean', ROUND(AVG(dia_chi_kiem_thu)),
       ROUND(AVG(acc_prefix), 2), ROUND(AVG(caught_prefix), 2), ROUND(AVG(fa_prefix), 2), ROUND(AVG(acc_both), 2), ROUND(AVG(caught_both), 2)
FROM per_seed
UNION ALL
SELECT 'sd', NULL, ROUND(STDDEV_SAMP(acc_prefix), 3), ROUND(STDDEV_SAMP(caught_prefix), 3), ROUND(STDDEV_SAMP(fa_prefix), 3), ROUND(STDDEV_SAMP(acc_both), 3), ROUND(STDDEV_SAMP(caught_both), 3)
FROM per_seed
)
ORDER BY CASE seed WHEN 'mean' THEN 2 WHEN 'sd' THEN 3 ELSE 1 END, seed
""")
print(q18.to_string(index=False))
t = df[df.split_s0 == "test"]
flag = 1 - t.starts_https_www
print("\nĐối chiếu pandas, split_s0: độ chính xác %.2f%%, bắt được %.2f%%" % (100 * (flag == t.phishing).mean(), 100 * flag[t.phishing == 1].mean()))

    seed  dia_chi_kiem_thu  do_chinh_xac_tien_to_pct  bat_duoc_tien_to_pct  bao_dong_gia_pct  do_chinh_xac_tien_to_hoac_duong_dan_pct  bat_duoc_tien_to_hoac_duong_dan_pct
split_s0             47074                     98.94                 97.51                 0                                    99.72                                99.34
split_s1             47074                     98.97                 97.58                 0                                     99.7                                99.31
split_s2             47074                     98.97                 97.58                 0                                    99.72                                99.35
split_s3             47074                     99.01                 97.69                 0                                    99.73                                99.36
split_s4             47074                     98.93                  97.5                 0                                    99.72            

**Nhận xét:** trên tập kiểm thử của năm seed, quy tắc tiền tố đạt 98,96% độ chính xác (độ lệch chuẩn 0,032) và bắt 97,57% địa chỉ lừa đảo. Quy tắc gộp đạt 99,72% (độ lệch chuẩn 0,008) và bắt 99,34%. Cả hai không báo động giả nào ở cả năm seed.
**Ý nghĩa cho câu hỏi 2:** mô hình nào ở Bước 6 cũng phải vượt 99,72% độ chính xác, hoặc ở mức 1% báo động giả đã chốt ở Bước 1 phải bắt được nhiều hơn 99,34% địa chỉ lừa đảo, thì mới có giá trị hơn hai điều kiện viết tay vốn không báo động giả nào. Các bài trên PhiUSIIL báo từ 95% tới 99,82% (Bước 1, mục 1.2), tức thấp hơn hoặc chỉ nhỉnh hơn mốc này.

## 3.19 Câu hỏi nghiên cứu 3: lối tắt trong các nhóm khó
Câu hỏi 3 hỏi điểm số còn lại bao nhiêu khi bỏ các lối tắt và khi gặp dữ liệu chưa thấy. Ô dưới dựng năm nhóm bằng `UNION ALL`: toàn bộ, trang chủ, TLD chưa thấy, trang chủ trên TLD chưa thấy, và tập kiểm thử của cách chia theo tên miền. Trên mỗi nhóm, ô chấm ba lối tắt: tiền tố, không dùng https, và chỉ số tương đồng dưới 100.
Cách đọc: nếu một lối tắt vẫn bắt gần hết địa chỉ lừa đảo với báo động giả 0% ở mọi nhóm, thì nó không phụ thuộc vào TLD hay tên miền mà phụ thuộc vào cách thu thập.

In [20]:
q19 = run("sql_19_rq3_rules_by_group", "Câu hỏi 3: quy tắc viết tay và chỉ số tương đồng trong các nhóm khó", """
WITH grouped AS (
    SELECT 'A all addresses' AS nhom, * FROM urls
    UNION ALL SELECT 'B homepage only', * FROM urls WHERE homepage_only = 1
    UNION ALL SELECT 'C unseen TLDs', * FROM urls WHERE tld_seen = 0
    UNION ALL SELECT 'D homepage only, unseen TLDs', * FROM urls WHERE homepage_only = 1 AND tld_seen = 0
    UNION ALL SELECT 'E domain split test, seed 0', * FROM urls WHERE gsplit_s0 = 'test'
)
SELECT nhom,
       COUNT(*)                                                                                   AS so_dia_chi,
       ROUND(100.0 * AVG(phishing), 1)                                                            AS lua_dao_pct,
       ROUND(100.0 * AVG(1 - starts_https_www) FILTER (WHERE phishing = 1), 2)                   AS tien_to_bat_duoc_pct,
       ROUND(100.0 * AVG(1 - starts_https_www) FILTER (WHERE phishing = 0), 2)                   AS tien_to_bao_dong_gia_pct,
       ROUND(100.0 * AVG(1 - IsHTTPS) FILTER (WHERE phishing = 1), 2)                            AS khong_https_bat_duoc_pct,
       ROUND(100.0 * AVG(CASE WHEN URLSimilarityIndex < 100 THEN 1 ELSE 0 END) FILTER (WHERE phishing = 1), 2) AS chi_so_bat_duoc_pct,
       ROUND(100.0 * AVG(CASE WHEN URLSimilarityIndex < 100 THEN 1 ELSE 0 END) FILTER (WHERE phishing = 0), 2) AS chi_so_bao_dong_gia_pct
FROM grouped
GROUP BY nhom
ORDER BY nhom
""")
print(q19.to_string(index=False))
hu = df[(df.homepage_only == 1) & (df.tld_seen == 0)]
print("\nĐối chiếu pandas, nhóm D: %d địa chỉ, quy tắc tiền tố bắt %.2f%% địa chỉ lừa đảo, báo động giả %.2f%%"
      % (len(hu), 100 * (1 - hu.starts_https_www)[hu.phishing == 1].mean(), 100 * (1 - hu.starts_https_www)[hu.phishing == 0].mean()))

                        nhom  so_dia_chi  lua_dao_pct  tien_to_bat_duoc_pct  tien_to_bao_dong_gia_pct  khong_https_bat_duoc_pct  chi_so_bat_duoc_pct  chi_so_bao_dong_gia_pct
             A all addresses      235370         42.7                 97.57                         0                     50.88                99.22                        0
             B homepage only      175363         23.1                 98.45                         0                     94.39                98.13                        0
               C unseen TLDs       97150         53.1                 97.69                         0                     47.34                99.51                        0
D homepage only, unseen TLDs       63784         28.6                 98.09                         0                     91.35                98.66                        0
 E domain split test, seed 0       43431         37.8                 96.91                         0                     64.08   

**Nhận xét:** quy tắc tiền tố bắt từ 96,91% tới 98,45% địa chỉ lừa đảo ở cả năm nhóm, chỉ số tương đồng bắt từ 98,13% tới 99,51%, và cả hai đều không báo động giả nào ở bất kỳ nhóm nào. Trên nhóm khó nhất (trang chủ trên TLD chưa thấy, 63.784 địa chỉ) quy tắc tiền tố vẫn bắt 98,09%.
Riêng quy tắc "không dùng https" thay đổi mạnh: bắt 50,88% trên toàn bộ nhưng 94,39% trên nhóm trang chủ, vì trang chủ lừa đảo hầu như không dùng https.
**Trả lời sơ bộ câu hỏi 3:** các lối tắt không yếu đi khi đổi TLD hay tên miền, nên chia theo TLD hay theo tên miền **không đủ** để bỏ chúng. Chỉ có bỏ hẳn các cột mang lối tắt (thang đặc trưng ở Bước 6) mới đo được phần điểm số còn lại.

## 3.20 Lưu các truy vấn ra tệp SQL và đóng kết nối
Ô dưới ghi 19 truy vấn vào `SQL/03_queries.sql`, mỗi truy vấn kèm tên và tiêu đề, rồi đóng kết nối tới `Outputs/phishing.duckdb`.
Cách đọc: dòng in ra phải báo 19 truy vấn và 19 tệp `sql_*.csv`.

In [21]:
with open(SQL_DIR + "03_queries.sql", "w", encoding="utf-8") as f:
    f.write("-- ADY201m, đồ án One-Line Rule: 19 truy vấn của notebook \"3. Data Analysis with SQL\".\n")
    f.write("-- Chạy trên Outputs/phishing.duckdb bằng DuckDB 1.0 trở lên (dùng ROLLUP, FILTER, QUALIFY, UNPIVOT, QUANTILE_CONT, regexp_extract).\n")
    f.write("-- Bảng raw_urls là tệp gốc, bảng urls là phishing_model_table.parquet của Bước 2, bảng tld_codes là tld_country_codes.csv.\n")
    for name, (title, sql) in QUERIES.items():
        f.write(f"\n-- {name}: {title}\n{sql};\n")
con.close()
saved = sorted(f for f in os.listdir(OUT) if f.startswith("sql_"))
print("Đã lưu SQL/03_queries.sql với", len(QUERIES), "truy vấn |", len(saved), "tệp sql_*.csv trong Outputs | cơ sở dữ liệu Outputs/phishing.duckdb")
print(saved)

Đã lưu SQL/03_queries.sql với 19 truy vấn | 19 tệp sql_*.csv trong Outputs | cơ sở dữ liệu Outputs/phishing.duckdb
['sql_01_overview.csv', 'sql_02_duplicates.csv', 'sql_03_two_classes.csv', 'sql_04_prefix_shapes.csv', 'sql_05_one_line_rules.csv', 'sql_06_similarity_bands.csv', 'sql_07_tld_rank.csv', 'sql_08_tld_type.csv', 'sql_09_countries.csv', 'sql_10_registered_domains.csv', 'sql_11_length_bands.csv', 'sql_12_length_quantiles.csv', 'sql_13_empty_pages.csv', 'sql_14_length_gap.csv', 'sql_15_split_domain_overlap.csv', 'sql_16_tld_seen_unseen.csv', 'sql_17_rq1_homepage_classes.csv', 'sql_18_rq2_rule_by_seed.csv', 'sql_19_rq3_rules_by_group.csv']


**Tóm tắt Bước 3.** 19 truy vấn SQL, mỗi truy vấn được đối chiếu bằng pandas, cho bốn kết luận dùng ở các bước sau:
- Hai lớp tách nhau gần như tuyệt đối ở tiền tố, đường dẫn, chỉ số tương đồng và độ trống của trang (mục 3.3 tới 3.6, 3.13). Không thứ nào trong số đó mô tả hành vi lừa đảo.
- Hai điều kiện viết tay đạt 99,72% độ chính xác trên tập kiểm thử của năm seed mà không báo động giả nào (mục 3.18). Đây là mốc của Bước 6.
- Chia ngẫu nhiên để 59% địa chỉ lừa đảo trong tập kiểm thử nằm trên tên miền đã gặp lúc huấn luyện (mục 3.15), và 20 tên miền lưu trữ miễn phí giữ 30,5% địa chỉ lừa đảo (mục 3.10).
- Các lối tắt đứng vững ở mọi nhóm khó, kể cả TLD chưa thấy và chia theo tên miền (mục 3.19), nên chỉ bỏ cột mới đo được điểm số thật.

Bước 4 dùng Python để khám phá sâu hơn các phân bố và quan hệ này.